In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset
from sklearn.utils.class_weight import compute_class_weight

# ==========================
# 1️⃣ Load Dataset
# ==========================

X = np.load("X_indiefake.npy")
y = np.load("y_indiefake.npy")

print("Original shape:", X.shape)

# Add channel dimension for CNN
X = X[:, np.newaxis, :, :]   # (N, 1, 60, 128)

# ==========================
# 2️⃣ Train-Test Split
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ==========================
# 3️⃣ Class Weights (Imbalance Handling)
# ==========================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float32)

# ==========================
# 4️⃣ Convert to Torch
# ==========================

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64)

# ==========================
# 5️⃣ CNN Model
# ==========================

class AudioCNN(nn.Module):
    def __init__(self):
        super(AudioCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),   # (16, 30, 64)

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # (32, 15, 32)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)    # (64, 7, 16)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ==========================
# 6️⃣ Setup Training
# ==========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AudioCNN().to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==========================
# 7️⃣ Training Loop
# ==========================

epochs = 15

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss/len(train_loader):.4f} "
          f"Train Acc: {train_acc:.2f}%")

# ==========================
# 8️⃣ Evaluation
# ==========================

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

# ==========================
# 9️⃣ Save Model
# ==========================

torch.save(model.state_dict(), "indiefake_cnn_model.pth")
print("\nModel saved as indiefake_cnn_model.pth")

Original shape: (16511, 60, 128)
Using device: cpu
Epoch [1/15] Loss: 0.5114 Train Acc: 74.61%
Epoch [2/15] Loss: 0.3237 Train Acc: 83.85%
Epoch [3/15] Loss: 0.2578 Train Acc: 87.55%
Epoch [4/15] Loss: 0.2184 Train Acc: 90.66%
Epoch [5/15] Loss: 0.1979 Train Acc: 91.87%
Epoch [6/15] Loss: 0.1859 Train Acc: 92.41%
Epoch [7/15] Loss: 0.1635 Train Acc: 93.22%
Epoch [8/15] Loss: 0.1489 Train Acc: 93.95%
Epoch [9/15] Loss: 0.1391 Train Acc: 94.50%
Epoch [10/15] Loss: 0.1289 Train Acc: 94.80%
Epoch [11/15] Loss: 0.1356 Train Acc: 94.52%
Epoch [12/15] Loss: 0.1158 Train Acc: 95.03%
Epoch [13/15] Loss: 0.1005 Train Acc: 95.91%
Epoch [14/15] Loss: 0.1069 Train Acc: 95.74%
Epoch [15/15] Loss: 0.0984 Train Acc: 95.95%

Classification Report:
              precision    recall  f1-score   support

           0     0.8922    0.9641    0.9267      1253
           1     0.9769    0.9288    0.9522      2050

    accuracy                         0.9422      3303
   macro avg     0.9345    0.9464    0.93

In [2]:
import numpy as np
import torch
import torch.nn as nn
import librosa

# ==========================
# 1️⃣ Model Definition (Same as Training)
# ==========================

class AudioCNN(nn.Module):
    def __init__(self):
        super(AudioCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ==========================
# 2️⃣ Load Trained Model
# ==========================

device = torch.device("cpu")

model = AudioCNN()
model.load_state_dict(torch.load("indiefake_cnn_model.pth", map_location=device))
model.to(device)
model.eval()

print("Model loaded successfully.")

# ==========================
# 3️⃣ Feature Extraction (Same as Training)
# ==========================

def extract_features(file_path):

    y, sr = librosa.load(file_path, sr=16000)

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)

    # Delta
    delta = librosa.feature.delta(mfcc)

    # Delta-Delta
    delta2 = librosa.feature.delta(mfcc, order=2)

    # Stack (60, frames)
    features = np.vstack([mfcc, delta, delta2])

    # Keep first 128 frames
    if features.shape[1] < 128:
        pad_width = 128 - features.shape[1]
        features = np.pad(features, ((0,0),(0,pad_width)), mode='constant')
    else:
        features = features[:, :128]

    return features


# ==========================
# 4️⃣ Run Prediction
# ==========================

audio_path = "test_audio1.mp3"

features = extract_features(audio_path)

# Add batch and channel dimension
features = np.expand_dims(features, axis=0)   # (1, 60, 128)
features = np.expand_dims(features, axis=1)   # (1, 1, 60, 128)

features = torch.tensor(features, dtype=torch.float32).to(device)

with torch.no_grad():
    outputs = model(features)
    probs = torch.softmax(outputs, dim=1)
    predicted = torch.argmax(probs, dim=1).item()

confidence = probs[0][predicted].item()

print("\nPrediction Result:")
if predicted == 0:
    print(f"REAL (Confidence: {confidence:.4f})")
else:
    print(f"FAKE (Confidence: {confidence:.4f})")

Model loaded successfully.


/home/saurav/project/college/code/venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Prediction Result:
FAKE (Confidence: 1.0000)


In [3]:
import os
import numpy as np
import torch
import torch.nn as nn
import librosa

# ==========================
# 1️⃣ Model Definition
# ==========================

class AudioCNN(nn.Module):
    def __init__(self):
        super(AudioCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ==========================
# 2️⃣ Load Model
# ==========================

device = torch.device("cpu")

model = AudioCNN()
model.load_state_dict(torch.load("indiefake_cnn_model.pth", map_location=device))
model.to(device)
model.eval()

print("Model loaded successfully.\n")


# ==========================
# 3️⃣ Feature Extraction
# ==========================

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    features = np.vstack([mfcc, delta, delta2])

    # Pad / Truncate to 128 frames
    if features.shape[1] < 128:
        pad_width = 128 - features.shape[1]
        features = np.pad(features, ((0,0),(0,pad_width)), mode='constant')
    else:
        features = features[:, :128]

    return features


# ==========================
# 4️⃣ Predict Folder
# ==========================

test_folder = "test_audio"

for file in os.listdir(test_folder):

    if file.endswith((".wav", ".mp3", ".ogg")):
        file_path = os.path.join(test_folder, file)

        features = extract_features(file_path)

        features = np.expand_dims(features, axis=0)
        features = np.expand_dims(features, axis=1)

        features = torch.tensor(features, dtype=torch.float32).to(device)

        with torch.no_grad():
            outputs = model(features)
            probs = torch.softmax(outputs, dim=1)
            predicted = torch.argmax(probs, dim=1).item()
            confidence = probs[0][predicted].item()

        label = "REAL" if predicted == 0 else "FAKE"

        print(f"{file} → {label} (Confidence: {confidence:.4f})")

Model loaded successfully.

ahrnish_test_audio_4.ogg → FAKE (Confidence: 1.0000)
ahrnish_test_audio_1.ogg → FAKE (Confidence: 1.0000)
ahrnish_test_audio_2.ogg → FAKE (Confidence: 0.9999)
fake_tts_test_audio_1.mp3 → FAKE (Confidence: 1.0000)
ahrnish_test_audio_3.ogg → FAKE (Confidence: 1.0000)


In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import librosa

# ==========================
# 1️⃣ Model Definition
# ==========================

class AudioCNN(nn.Module):
    def __init__(self):
        super(AudioCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ==========================
# 2️⃣ Load Model
# ==========================

device = torch.device("cpu")

model = AudioCNN()
model.load_state_dict(torch.load("indiefake_cnn_model.pth", map_location=device))
model.to(device)
model.eval()

print("Model loaded successfully.\n")


# ==========================
# 3️⃣ Feature Extraction
# ==========================

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    features = np.vstack([mfcc, delta, delta2])

    # Pad / Truncate to 128 frames
    if features.shape[1] < 128:
        pad_width = 128 - features.shape[1]
        features = np.pad(features, ((0,0),(0,pad_width)), mode='constant')
    else:
        features = features[:, :128]

    return features


# ==========================
# 4️⃣ Predict Folder
# ==========================

test_folder = "test_audio"

for file in os.listdir(test_folder):

    if file.endswith((".wav", ".mp3", ".ogg")):
        file_path = os.path.join(test_folder, file)

        features = extract_features(file_path)

        features = np.expand_dims(features, axis=0)
        features = np.expand_dims(features, axis=1)

        features = torch.tensor(features, dtype=torch.float32).to(device)

        with torch.no_grad():
            outputs = model(features)
            probs = torch.softmax(outputs, dim=1)
            predicted = torch.argmax(probs, dim=1).item()
            confidence = probs[0][predicted].item()

        label = "REAL" if predicted == 0 else "FAKE"

        print(f"{file} → {label} (Confidence: {confidence:.4f})")

Model loaded successfully.



/home/saurav/project/college/code/venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


voice_4.wav → FAKE (Confidence: 0.9887)
voice_1.wav → FAKE (Confidence: 0.9999)
voice_2.wav → FAKE (Confidence: 0.9890)
voice_3.wav → FAKE (Confidence: 1.0000)
voice_5.wav → FAKE (Confidence: 0.8078)


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import librosa

# ==========================
# 1️⃣ Model Definition
# ==========================

class AudioCNN(nn.Module):
    def __init__(self):
        super(AudioCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ==========================
# 2️⃣ Load Model
# ==========================

device = torch.device("cpu")

model = AudioCNN()
model.load_state_dict(torch.load("indiefake_cnn_model.pth", map_location=device))
model.to(device)
model.eval()

print("Model loaded successfully.\n")


# ==========================
# 3️⃣ Feature Extraction
# ==========================

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    features = np.vstack([mfcc, delta, delta2])

    # Pad / Truncate to 128 frames
    if features.shape[1] < 128:
        pad_width = 128 - features.shape[1]
        features = np.pad(features, ((0,0),(0,pad_width)), mode='constant')
    else:
        features = features[:, :128]

    return features


# ==========================
# 4️⃣ Predict Folder
# ==========================

test_folder = "test_audio"

for file in os.listdir(test_folder):

    if file.endswith((".wav", ".mp3", ".ogg")):
        file_path = os.path.join(test_folder, file)

        features = extract_features(file_path)

        features = np.expand_dims(features, axis=0)
        features = np.expand_dims(features, axis=1)

        features = torch.tensor(features, dtype=torch.float32).to(device)

        with torch.no_grad():
            outputs = model(features)
            probs = torch.softmax(outputs, dim=1)
            predicted = torch.argmax(probs, dim=1).item()
            confidence = probs[0][predicted].item()

        label = "REAL" if predicted == 0 else "FAKE"

        print(f"{file} → {label} (Confidence: {confidence:.4f})")

Model loaded successfully.

voice_57.wav → FAKE (Confidence: 1.0000)
voice_61.wav → FAKE (Confidence: 0.9899)
voice_60.wav → FAKE (Confidence: 0.9998)
voice_59.wav → FAKE (Confidence: 0.9869)
voice_58.wav → FAKE (Confidence: 1.0000)
